In [1]:
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from tensordict.nn.distributions import NormalParamExtractor
from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer, TensorDictReplayBuffer, LazyTensorStorage, SamplerWithoutReplacement
from torchrl.modules import ProbabilisticActor, ValueOperator, TanhNormal, MultiAgentMLP
from torchrl.objectives import ClipPPOLoss, ValueEstimators
from torchrl.objectives.value import GAE
from torch import multiprocessing
from torchrl.envs import RewardSum, TransformedEnv, check_env_specs, Transform

from torchrl.envs import PettingZooWrapper
from momentum_env.momentum_env import Momentum

In [2]:
env = Momentum()
group_map = {"team0": ['player1', 'player2', 'player3'],
            "team1": ['player4', 'player5', 'player6']}

In [3]:
env = PettingZooWrapper(env, group_map=group_map)

In [4]:
is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)

device = torch.device("cpu")

In [5]:
team0_policy_net = torch.nn.Sequential(
    MultiAgentMLP(
        n_agent_inputs=env.observation_spec["team0", "observation"].shape[-1],
        n_agent_outputs = 2 * 4,
        n_agents = 3,
        centralised = False,
        share_params = False,
        device=device,
        depth=2,
        num_cells = 128,
        activation_class = torch.nn.Tanh,
    ),
    NormalParamExtractor(),
).to(device)

In [6]:
team0_policy_module = TensorDictModule(
    team0_policy_net,
    in_keys = [("team0", "observation")],
    out_keys = [("team0", "loc"), ("team0", "scale")],
).to(device)

In [7]:
team0_policy = ProbabilisticActor(
    module = team0_policy_module,
    spec=env.action_spec["team0", "action"],
    in_keys = [("team0", "loc"), ("team0", "scale")],
    out_keys = [env.action_keys[0]],
    distribution_class = TanhNormal,
    distribution_kwargs = {
        "min": env.action_spec[env.action_keys[0]].space.low[0],
        "max": env.action_spec[env.action_keys[0]].space.high[0],
    },
    return_log_prob = True,
    log_prob_key = ("team0", "sample_log_prob"),
).to(device)

In [8]:
team1_policy_net = torch.nn.Sequential(
    MultiAgentMLP(
        n_agent_inputs=env.observation_spec["team1", "observation"].shape[-1],
        n_agent_outputs = 2 * 4,
        n_agents = 3,
        centralised = False,
        share_params = False,
        device=device,
        depth=2,
        num_cells = 128,
        activation_class = torch.nn.Tanh,
    ),
    NormalParamExtractor(),
).to(device)

In [9]:
team1_policy_module = TensorDictModule(
    team1_policy_net,
    in_keys = [("team1", "observation")],
    out_keys = [("team1", "loc"), ("team1", "scale")],
).to(device)

In [10]:
team1_policy = ProbabilisticActor(
    module = team1_policy_module,
    spec=env.action_spec["team1", "action"],
    in_keys = [("team1", "loc"), ("team1", "scale")],
    out_keys = [env.action_keys[1]],
    distribution_class = TanhNormal,
    distribution_kwargs = {
        "min": env.action_spec[env.action_keys[1]].space.low[0],
        "max": env.action_spec[env.action_keys[1]].space.high[0],
    },
    return_log_prob = True,
    log_prob_key = ("team1", "sample_log_prob"),
).to(device)

In [12]:
team0_policy = torch.load("team0_v2.pt")
team1_policy = torch.load("team0_v2.pt")

In [13]:
policy = TensorDictSequential(
   team0_policy,
   team1_policy,
)

In [19]:
td = env.reset()
for i in range(100):
    td = policy(td)